# 21 — Recalibrate + re-tune the blend against the flip-TTA'd 6-variant ensemble

**What notebook 20 found (2026-09-10)**: flip-TTA on the `rung4_augment`
checkpoints (average of each checkpoint's prediction with its L-R-flipped
counterpart, principled because these checkpoints were trained with flip
augmentation) beat the plain augment OOF in 5/5 seeds individually, and
improved the already-adopted 6-variant ensemble on the pre-registered
comparison (log loss 0.3978→0.3976, ECE 0.0305→0.0293, AUROC flat). Per
the pre-registered decision rule, flip-TTA is **adopted** for the augment
checkpoints — `rung4_augment_tta_oof_seed{42..46}.npy` (saved by notebook
20) replaces `rung4_augment_oof_seed{s}.npy` in the production
composition.

Notebook 19's calibration (T_cnn=0.58, T_baseline=0.88, w=0.53) was fit
against the 6-variant ensemble's *plain*-augment CNN signal. Since the
"CNN side" of the blend has changed again (augment member swapped for its
TTA'd version), those parameters need to be **re-fit** once more — this
notebook is notebooks 15/17/19's exact method, applied to this composition.

**Per-repeat "CNN" signal, redefined again**: for each of the 5 seeds,
this notebook's `cnn_oof_repeats[i]` is the average of that seed's 6
variant OOF arrays — rung3, familybias, lrsched, **augment_tta** (in place
of plain augment), classweight, fixedepoch. The classical-baseline side is
unchanged (reuses `baseline_oof_seed{42..46}.npy`, already computed and
cached by notebook 15 — no need to recompute).

Per the roadmap (`project_dat_parkinson_strategic_roadmap.md`), items 1-7
are now all addressed once this notebook's result is in — the next step
after this is implementation, not further local-CV exploration.

**Data handling**: loads real row-level labels and OOF prediction arrays,
so per the AI-assistant data rule (`README.md`) this is **[RUN ME]** — run
it yourself, share back only the printed aggregate numbers. CPU-only, no
GPU, no volume cache — seconds to low tens of seconds for the grid
search.

In [1]:
# [RUN ME] -- loads real row-level labels + existing OOF prediction arrays
# (5 non-augment variants + the augment family's flip-TTA'd OOF from
# notebook 20 + the per-seed classical baseline OOF notebook 15 already
# cached). CPU-only, no GPU, no volume cache -- self-contained, does not
# assume any earlier cell/notebook ran in this kernel session.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

import config
import evaluate
import model

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)
uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
y_true = np.array(labels)

repeat_seeds = list(range(config.SEED, config.SEED + 5))
VARIANT_PREFIXES = ["rung3", "rung4_familybias", "rung4_lrsched", "rung4_augment_tta",
                     "rung4_classweight", "rung4_fixedepoch"]

# per-seed 6-variant-ensembled "CNN" signal -- this replaces notebook 19's
# plain-augment 6-variant ensemble with the flip-TTA'd augment one.
cnn_oof_repeats = []
for s in repeat_seeds:
    variant_arrays = [np.load(config.DATA_PROCESSED / f"{prefix}_oof_seed{s}.npy") for prefix in VARIANT_PREFIXES]
    cnn_oof_repeats.append(np.mean(variant_arrays, axis=0))

# classical baseline OOF, per seed -- reuse notebook 15's cache if present,
# else recompute the same way it did (should already exist on disk).
baseline_feat_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")
baseline_feat_df = baseline_feat_df.set_index(config.UID_COLUMN).loc[uids].reset_index()
baseline_X = baseline_feat_df[["abs_asym", "striatal_ratio"]].to_numpy()
baseline_y = baseline_feat_df[config.TARGET_COLUMN].to_numpy()
baseline_family = baseline_feat_df["inplane_family"].to_numpy()
assert np.array_equal(baseline_y, y_true), "baseline_features.csv row order must match labels_df merge"

baseline_oof_repeats = []
for seed in repeat_seeds:
    cache_path = config.DATA_PROCESSED / f"baseline_oof_seed{seed}.npy"
    if cache_path.exists():
        baseline_oof_repeats.append(np.load(cache_path))
        continue
    baseline_oof = np.zeros(len(uids))
    folds = evaluate.make_folds(baseline_y, baseline_family, n_splits=config.N_FOLDS, random_state=seed)
    for train_idx, test_idx in folds:
        pipeline = model.build_combat_baseline()
        pipeline.fit(baseline_X[train_idx], baseline_y[train_idx], baseline_family[train_idx])
        baseline_oof[test_idx] = pipeline.predict_proba(baseline_X[test_idx], baseline_family[test_idx])[:, 1]
    np.save(cache_path, baseline_oof)
    baseline_oof_repeats.append(baseline_oof)

print(f"{len(repeat_seeds)} repeats ready: 6-variant-ensembled CNN (augment=flip-TTA) + seed-matched baseline OOF.")

5 repeats ready: 6-variant-ensembled CNN (augment=flip-TTA) + seed-matched baseline OOF.


In [2]:
# [RUN ME] (no new data access -- uses the arrays built above).
# LOFO-validated joint grid search over (T_cnn, T_baseline, w) in logit
# space -- identical method to notebooks 15/17/19, applied to the
# flip-TTA'd 6-variant CNN signal.
T_GRID = np.arange(0.5, 2.01, 0.1)
W_GRID = np.arange(0.0, 1.01, 0.05)
EPS = 1e-6


def to_logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))


def calibrated_blend(cnn_p, baseline_p, t_cnn, t_baseline, w):
    combined_logit = w * (to_logit(cnn_p) / t_cnn) + (1 - w) * (to_logit(baseline_p) / t_baseline)
    return 1.0 / (1.0 + np.exp(-combined_logit))


def fast_log_loss(y, p):
    p = np.clip(p, EPS, 1 - EPS)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


cnn_logits = [to_logit(oof) for oof in cnn_oof_repeats]
baseline_logits = [to_logit(oof) for oof in baseline_oof_repeats]

honest_scores = []
selected_params = []
for held_out_i in range(len(repeat_seeds)):
    selection = [i for i in range(len(repeat_seeds)) if i != held_out_i]
    best = None
    for t_cnn in T_GRID:
        cnn_scaled = [cnn_logits[i] / t_cnn for i in selection]
        for t_base in T_GRID:
            base_scaled = [baseline_logits[i] / t_base for i in selection]
            for w in W_GRID:
                mean_ll = np.mean([
                    fast_log_loss(y_true, 1.0 / (1.0 + np.exp(-(w * cs + (1 - w) * bs))))
                    for cs, bs in zip(cnn_scaled, base_scaled)
                ])
                if best is None or mean_ll < best[0]:
                    best = (mean_ll, float(t_cnn), float(t_base), float(w))
    _, t_cnn, t_base, w = best
    held_out_probs = calibrated_blend(cnn_oof_repeats[held_out_i], baseline_oof_repeats[held_out_i], t_cnn, t_base, w)
    held_out_score = evaluate.log_loss_score(y_true, held_out_probs)
    honest_scores.append(held_out_score)
    selected_params.append((t_cnn, t_base, w))
    print(f"  held-out repeat {held_out_i} (seed={repeat_seeds[held_out_i]}): "
          f"selected T_cnn={t_cnn:.2f}, T_baseline={t_base:.2f}, w={w:.2f} on the other 4, "
          f"scored {held_out_score:.4f} on this one")

honest_scores = np.array(honest_scores)
print(f"\nLOFO calibrated-blend (6-variant ensemble, augment=flip-TTA): mean={honest_scores.mean():.4f}, "
      f"sd={honest_scores.std(ddof=1):.4f}")
print("for comparison -- notebook 19's plain-augment 6-variant LOFO calibrated-blend: mean=0.3740 sd=0.0058")
print("for comparison -- notebook 20's uncalibrated flip-TTA'd 6-variant ensemble (pooled): log loss=0.3976")

  held-out repeat 0 (seed=42): selected T_cnn=0.70, T_baseline=0.70, w=0.65 on the other 4, scored 0.3807 on this one
  held-out repeat 1 (seed=43): selected T_cnn=0.50, T_baseline=1.00, w=0.45 on the other 4, scored 0.3761 on this one
  held-out repeat 2 (seed=44): selected T_cnn=0.80, T_baseline=0.50, w=0.75 on the other 4, scored 0.3678 on this one
  held-out repeat 3 (seed=45): selected T_cnn=0.50, T_baseline=1.00, w=0.45 on the other 4, scored 0.3675 on this one
  held-out repeat 4 (seed=46): selected T_cnn=0.80, T_baseline=0.50, w=0.75 on the other 4, scored 0.3772 on this one

LOFO calibrated-blend (6-variant ensemble, augment=flip-TTA): mean=0.3739, sd=0.0059
for comparison -- notebook 19's plain-augment 6-variant LOFO calibrated-blend: mean=0.3740 sd=0.0058
for comparison -- notebook 20's uncalibrated flip-TTA'd 6-variant ensemble (pooled): log loss=0.3976


In [3]:
# [RUN ME] (no new data access -- uses the arrays built above).
# Final candidate params (mean of the 5 LOFO-selected triples). NOTE: as in
# notebooks 15/17/19, this pooled-ensemble application has mild optimism
# relative to the honest per-repeat LOFO mean above -- trust the LOFO
# mean/sd for the go/no-go decision, use this only as the actual recipe to
# implement.
final_t_cnn = float(np.mean([p[0] for p in selected_params]))
final_t_baseline = float(np.mean([p[1] for p in selected_params]))
final_w = float(np.mean([p[2] for p in selected_params]))
print(f"final params (mean of 5 LOFO-selected triples): "
      f"T_cnn={final_t_cnn:.2f}, T_baseline={final_t_baseline:.2f}, w={final_w:.2f}")

cnn_ensemble_oof = np.mean(cnn_oof_repeats, axis=0)
baseline_ensemble_oof = np.mean(baseline_oof_repeats, axis=0)
candidate_oof = calibrated_blend(cnn_ensemble_oof, baseline_ensemble_oof, final_t_cnn, final_t_baseline, final_w)

candidate_scores = evaluate.combined_score(y_true, candidate_oof)
print(f"\npooled candidate (calibrated, re-tuned blend, 6-variant ensemble, augment=flip-TTA): "
      f"log loss={candidate_scores['log_loss']:.4f}  AUROC={candidate_scores['auroc']:.4f}  "
      f"ECE={candidate_scores['ece']:.4f}")
print("real leaderboard (first submission): log loss=0.4648 AUROC=0.8796")

final params (mean of 5 LOFO-selected triples): T_cnn=0.66, T_baseline=0.74, w=0.61

pooled candidate (calibrated, re-tuned blend, 6-variant ensemble, augment=flip-TTA): log loss=0.3642  AUROC=0.9178  ECE=0.0378
real leaderboard (first submission): log loss=0.4648 AUROC=0.8796


**What we're looking for:** with flip-TTA adopted for the augment
checkpoints (notebook 20), does re-fitting calibration + blend weight
against this composition (instead of carrying over notebook 19's
plain-augment params) give a further, honest LOFO improvement -- and
where do T_cnn/T_baseline/w land this time?

**What we found:**
```
held-out repeat 0 (seed=42): T_cnn=0.70, T_baseline=0.70, w=0.65 -> 0.3807
held-out repeat 1 (seed=43): T_cnn=0.50, T_baseline=1.00, w=0.45 -> 0.3761
held-out repeat 2 (seed=44): T_cnn=0.80, T_baseline=0.50, w=0.75 -> 0.3678
held-out repeat 3 (seed=45): T_cnn=0.50, T_baseline=1.00, w=0.45 -> 0.3675
held-out repeat 4 (seed=46): T_cnn=0.80, T_baseline=0.50, w=0.75 -> 0.3772

LOFO calibrated-blend (augment=flip-TTA): mean=0.3739, sd=0.0059
notebook 19's plain-augment LOFO calibrated-blend:  mean=0.3740, sd=0.0058

final params (mean of 5 triples): T_cnn=0.66, T_baseline=0.74, w=0.61
pooled candidate: log loss=0.3642  AUROC=0.9178  ECE=0.0378
real leaderboard (first submission): log loss=0.4648 AUROC=0.8796
```
**The LOFO delta is -0.0001 -- inside the sd (0.0059), i.e. flat.** This is
the honest number to trust (the pooled 0.3642 vs. notebook 19's pooled
0.3646 looks like a win but both carry the same mild-optimism caveat this
project has flagged since notebook 15 -- comparing pooled-to-pooled here
would repeat exactly the mistake that caveat exists to prevent). Parameter
selection is also less coherent this time: 3 distinct points across 5
folds (2/5, 2/5, 1/5) vs. notebook 19's two clusters and notebook 17's
perfect 5/5 unanimity -- consistent with a flat region of the loss surface
rather than a real shift, which fits a null result.

**Mechanistic read, tying this back to notebook 20**: TTA's own effect was
real but small and acted mainly through ECE (0.0305→0.0293 at the raw
ensemble level, item 20's finding) -- and calibration's whole job is to
correct exactly that kind of miscalibration. Once the LOFO-fit blend
recalibrates either composition, it absorbs the small edge TTA was
providing on its own; there's nothing left for it to add downstream. Two
real, honestly-measured findings, not a contradiction: item 7's own
pre-registered gate (raw ensemble) was a real, if tiny, win; this
recalibration shows that win doesn't survive into the number that
actually decides the shipped recipe.

**Decision (user confirmed): apply this project's own step-6 gate rule
(`structuring-ml-projects`: "a win smaller than the measured noise floor
is not a win") at the level that matters -- the LOFO calibrated blend, not
the raw ensemble delta.** By that bar, flip-TTA does not clear the gate:
0.3739 vs. 0.3740 is noise, and TTA has a real, non-free cost (~doubles
in-container inference time for the augment checkpoints, 25 of 150 -- the
one resource repeatedly flagged as actually tight). **Flip-TTA dropped.**
Final adopted recipe reverts to notebook 19's: **plain-augment 6-variant
ensemble (150 checkpoints, no TTA) + calibrated blend T_cnn=0.58,
T_baseline=0.88, w=0.53, LOFO mean 0.3740.** Flip-TTA is logged as a
validated-but-not-adopted-in-the-final-recipe finding (step 7: negative
result, not discarded) -- the inference code path stays available in
notebook 20 / `rung4_augment_tta_oof_seed{42..46}.npy` if ever revisited,
just not wired into `submission_src/main.py`.

**Correction**: roadmap items 1-5 and 7 are addressed by this point; item
6 (bounded architecture check, e.g. a 2D-slab CNN as an additional
ensemble member) was never actually run and remains open -- no notebook
exists for it. Whether to spend time on it before implementation, given
the diminishing-returns pattern and the ~6-day/2-3-submission budget, is
an open call for the next strategic review (see
`project_dat_parkinson_strategic_roadmap.md`).

Next step per the roadmap: wire the notebook-19 recipe into
`src/submission.py::combine_predictions` and `submission_src/main.py`
(6-variant ensemble, plain augment, no TTA inference path needed), extend
`scripts/build_submission_assets.py`'s checkpoint packaging, address the
sklearn-pickle risk, rebuild + smoke-test, before spending a real
submission.